![Kenya Food Price Early Warning System](images/banner.png)

# Kenya Food Price Early Warning System


Forecasting staple food prices across Kenyan markets to give farmers, traders, and food security actors the early signal they currently lack.

## 1. Business Understanding

## 1.1 Background

Food prices in Kenya are highly seasonal and can change sharply across markets. Staples such as maize and beans are affected by harvest cycles, rainfall, supply conditions, and limited access to timely market information.

For farmers, this creates a difficult decision: sell early and risk missing a better price, or wait and risk a price drop. Traders and institutions face a similar problem when deciding when to buy, store, release, or distribute food.

The problem is not a lack of historical data. Kenya has publicly available market price data, but this information is mainly used to understand what has already happened.

This project aims to turn that historical information, together with weather data, into a forward-looking system that shows **where prices are likely to move and when unusual price changes may be starting.**

## 1.2 Problem Statement

Farmers, traders, and food security institutions often make decisions using current or historical prices rather than reliable forecasts of what may happen next.

This can lead to poor selling and buying decisions, increased inventory risk, and delayed responses to food price shocks. Institutions may only act once a shortage or price increase is already visible.

The core problem is therefore not the absence of data. It is the lack of a system that combines historical prices and external factors such as weather to produce **market-specific forecasts and early warnings.**

## 1.3 Business Objectives

The project aims to:

1. **Provide forward-looking price visibility**  
   Forecast commodity prices 2–3 months ahead for individual markets.

2. **Detect emerging price shocks**  
   Identify when actual prices begin to move significantly away from expected prices.

3. **Improve access to market intelligence**  
   Present forecasts and trends through a simple interactive dashboard that non-technical users can understand.

4. **Support institutional decisions**  
   Provide quantitative signals that can support decisions around food reserves, subsidies, procurement, and humanitarian response.

5. **Build a reproducible system**  
   Create an automated pipeline from data collection and processing to forecasting and dashboard deployment.

### 1.4 Stakeholder Analysis

The system is designed for people who make decisions around food prices, from farmers and traders to government and humanitarian organizations.

**Smallholder Farmers and Cooperatives**
Need to decide when to sell, how much to sell, and whether holding stock could lead to a better return. Price forecasts can give them greater visibility into upcoming market conditions.

**Traders and Market Intermediaries**
Need to decide when and where to buy, store, and sell commodities. Forecasts can help them manage inventory and reduce the risk of buying at a peak or holding through a price decline.

**County Agricultural Offices and NDMA**
Need early signals of localized food price stress. Market-level forecasts and anomaly alerts can help them identify emerging risks before they become wider food security problems.

**NGOs and Humanitarian Organizations**
Need to plan procurement and cash-based interventions efficiently. Earlier visibility into price movements can help them act before rising prices reduce the purchasing power of their interventions.

**National Cereals and Produce Board**
Needs to make better decisions around strategic reserves, procurement, and price stabilization. Forecasts can provide an additional signal when deciding when to buy or release stock.

**Urban Consumers and Low-Income Households**
Are highly sensitive to changes in staple food prices. Earlier information about expected price movements can help households and organizations supporting them prepare for potential increases.

**Food Processors and Millers**
Need predictable input costs to manage production and pricing. Forward-looking commodity prices can support better procurement and planning.

Across these groups, the common need is **timely, market-specific information about where food prices are heading**. The forecasting and anomaly detection components are designed to provide that signal.


## 1.5 Business Success Criteria

The project will be considered successful if:

- Forecasts perform better than a simple baseline such as the last observed price or seasonal average.
- The anomaly detection system identifies meaningful price shocks without producing excessive false alarms.
- A non-technical user can select a market and commodity and quickly understand the expected price, forecast range, and alert status.
- The entire data pipeline is reproducible and can be refreshed without significant manual work.
- The system uses reliable and accessible public data sources so it can be extended beyond the capstone.

## 1.6 Data Mining Goals

The data science work will focus on five main tasks:

1. Build a **Prophet forecasting model** as the baseline.
2. Build an **LSTM model** for comparison.
3. Combine WFP Kenya food price data with NASA POWER weather data using market location and date.
4. Build a **residual-based anomaly detection system** to identify unusual price movements.
5. Compare the models using **MAE and MAPE** across commodities and markets.

The final forecasts and alerts will be presented through an interactive **Streamlit dashboard**.

## 1.7 Data Mining Success Criteria

The technical implementation will be considered successful if:

- The forecasting models outperform the chosen naive baseline across most market-commodity combinations.
- Forecast accuracy is evaluated using MAE and MAPE.
- The anomaly detector identifies historical price shocks while limiting false alerts.
- The price and weather datasets can be joined with minimal data loss.
- The pipeline can run end-to-end with minimal manual intervention.
- The deployed dashboard works reliably for the selected markets and commodities.

## 1.8 Hypotheses Guiding the Analysis

The analysis will test six hypotheses:

- **H1:** Maize prices follow a seasonal pattern around harvest periods.
- **H2:** Rainfall has a measurable lagged relationship with future commodity prices.
- **H3:** Price patterns differ significantly between markets.
- **H4:** Maize and bean prices show a relationship because they are commonly used as staple food substitutes.
- **H5:** Price volatility increases during drought periods.
- **H6:** Changes in wholesale prices are reflected in retail prices within one to two weeks.

These hypotheses will guide the exploratory analysis, feature engineering, modelling, and evaluation throughout the project.

## 2. Data Understanding

This section is highly  focused on understanding the data before cleaning, joining, and modelling.

The project uses two main data sources: the **WFP Kenya Food Prices dataset**, sourced through HDX, and daily weather data from the **NASA POWER API**.

The goal is to understand what each dataset contains, assess its quality, and look for early patterns that can help test the hypotheses from the Business Understanding phase.


### 2.1 Data Collection

The project uses two publicly accessible data sources.

**WFP Kenya Food Prices (HDX)**
Provides historical commodity prices by market, commodity, and date. This is the primary source for the target variable used in forecasting.

**NASA POWER API**
Provides daily weather observations, including rainfall and temperature. These variables are used as external features to test whether weather conditions can improve price forecasts.

The extraction date and source versions are recorded to ensure that the analysis can be reproduced using the same data snapshot.


In [ ]:
# core libraries and notebook display settings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests # used for routing and fetching our data and later in in demo(front end)
from datetime import datetime



In [ ]:
# set display settings for the notebook

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# record the date at which the last data was pulled.

extraction_date = datetime.now().strftime("%Y-%m-%d")

print(f"Data extraction date recorded: {extraction_date}")


In [ ]:
# Load the Kenya food prices dataset. Add support for Kaggle and local loading seamlessly.

DATA_URL = (
    "https://data.humdata.org/dataset/"
    "e0d3fba6-f9a2-45d7-b949-140c455197ff/"
    "resource/517ee1bf-2437-4f8c-aa1b-cb9925b9d437/"
    "download/wfp_food_prices_ken.csv"
)

# define a filename
FILENAME = "wfp_food_prices_ken.csv"


def load_food_prices():
    """
    Load the Kenya food prices dataset.

    The function automatically checks for a Kaggle copy,
    a previously saved local copy, or the original source URL.

    It returns:
        pandas.DataFrame
            Raw Kenya food prices dataset.
    """

    # check if the notebook is running on Kaggle
    kaggle_root = "/kaggle/input"

    if os.path.exists(kaggle_root):

        # search for the dataset inside the Kaggle input directory
        for root, _, files in os.walk(kaggle_root):

            if FILENAME in files:

                kaggle_path = os.path.join(root, FILENAME)

                try:
                    prices_raw = pd.read_csv(kaggle_path)

                    print(f"Loaded dataset from Kaggle: {kaggle_path}")

                    return prices_raw

                except Exception as error:
                    print(f"Kaggle file could not be read: {error}")

    # check if a previously downloaded local copy exists
    if os.path.exists(FILENAME):

        try:
            prices_raw = pd.read_csv(FILENAME)

            print(f"Loaded local dataset: {FILENAME}")

            return prices_raw

        except Exception as error:
            print(f"Local file could not be read: {error}")

    # try downloading from the source URL
    print("Dataset not found locally. Trying the source URL...")

    try:
        response = requests.get(
            DATA_URL,
            timeout=30
        )

        response.raise_for_status()

        # save the downloaded file locally for future use
        with open(FILENAME, "wb") as file:
            file.write(response.content)

        prices_raw = pd.read_csv(FILENAME)

        print(f"Dataset downloaded and saved as '{FILENAME}'")

        return prices_raw

    except Exception as error:
        print(f"Download failed: {error}")

    # final fallback to the local file
    if os.path.exists(FILENAME):

        print("Using the previously saved local dataset.")

        return pd.read_csv(FILENAME)

    raise FileNotFoundError(
        "Could not load the Kenya food prices dataset. "
        "Check your internet connection or provide a local copy."
    )


In [ ]:
# Load the dataset
prices_raw = load_food_prices()

# Output the first 5 rows
prices_raw.head()

In [ ]:
# drop the units/description row if present, then fix dtypes 

def clean_price_data(prices_raw):
    """Clean and convert data types in the raw price dataset."""

    # Remove the units/description row if present
    if prices_raw.iloc[0].astype(str).str.startswith("#").any():

        prices = prices_raw.iloc[1:].reset_index(drop=True)

    else:

        prices = prices_raw.copy()

    # Convert date and price columns to the correct data types
    prices["date"] = pd.to_datetime(
        prices["date"],
        errors="coerce"
    )

    prices["price"] = pd.to_numeric(
        prices["price"],
        errors="coerce"
    )

    # Convert USD price if the column exists
    if "usdprice" in prices.columns:

        prices["usdprice"] = pd.to_numeric(
            prices["usdprice"],
            errors="coerce"
        )

    return prices


In [ ]:
prices = clean_price_data(prices_raw)

In [ ]:
print(f"Rows: {prices.shape[0]}, Columns: {prices.shape[1]}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")

In [ ]:
# Get a sample of daily weather data from the NASA POWER API.
def get_weather_sample(
    latitude,
    longitude,
    start="20240101",
    end="20240131"
):
    """

    It returns:
        dict
            Weather parameters returned by the API.
    """

    power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"

    params = {
        "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M",
        "community": "ag",
        "longitude": longitude,
        "latitude": latitude,
        "start": start,
        "end": end,
        "format": "JSON",
    }

    # ping the url and pass a timeout to avoid out of control requests
    response = requests.get(
        power_url,
        params=params,
        timeout=30
    )

    # raise an error if the API request failed
    response.raise_for_status()
    print(f" status code {response.status_code}")


    power_sample = response.json()

    return power_sample["properties"]["parameter"]




In [ ]:
# sample call for Nairobi to confirm the API and response structure
nairobi_lat, nairobi_lon = -1.2864, 36.8172

sample_params = get_weather_sample(
    latitude=nairobi_lat,
    longitude=nairobi_lon
)


In [ ]:

print("Parameters returned:", list(sample_params.keys()))
print("Sample PRECTOTCORR values:", list(sample_params["PRECTOTCORR"].items())[:5])

### NASA POWER Weather Variables

The NASA POWER API provides several weather variables that can be used to understand conditions that may influence commodity prices.

| Parameter | Description | Unit |
|---|---|---|
| `T2M` | Average air temperature measured 2 meters above the surface | °C |
| `T2M_MAX` | Maximum air temperature measured at 2 meters | °C |
| `T2M_MIN` | Minimum air temperature measured at 2 meters | °C |
| `T2M_RANGE` | Difference between the daily maximum and minimum temperature | °C |
| `RH2M` | Relative humidity measured at 2 meters | % |
| `WS2M` | Average wind speed measured at 2 meters | m/s |
| `PS` | Atmospheric pressure at the surface | kPa |
| `PRECTOTCORR` | Bias-corrected total precipitation | mm/day |
| `ALLSKY_SFC_SW_DWN` | Total solar radiation reaching the surface under all sky conditions | kW-hr/m²/day |

For this project, **rainfall and temperature** are the primary weather variables of interest because they have the strongest potential relationship with agricultural production and future commodity prices.

### 2.2 Data Description

Before combining the datasets, we first need to understand how each one is structured.

The **price dataset** contains one observation for a specific commodity, market, and date.

The **weather dataset** contains daily weather observations for a specific geographic location.

This structure allows the two datasets to be joined using **market location and date**.


In [ ]:
# column level summary: dtype, uniqueness, missingness
data_dictionary = pd.DataFrame({
    "column": prices.columns,

    "dtype": [str(prices[col].dtype) for col in prices.columns],

    "n_unique": [prices[col].nunique() for col in prices.columns],

    "n_missing": [prices[col].isna().sum() for col in prices.columns],

    "pct_missing": [(prices[col].isna().mean() * 100).round(2) for col in prices.columns],
})

data_dictionary

In [ ]:
# scale and spread of the two numeric price fields

prices[["price", "usdprice"]].describe()

In [ ]:
# check for duplicate rows at the expected grain

grain_columns = ["date", "market", "commodity", "pricetype"]

duplicate_count = prices.duplicated(subset=grain_columns).sum()

In [ ]:
print(f"Duplicate rows at (date, market, commodity, pricetype) grain: {duplicate_count}")
print(f"Unique markets: {prices['market'].nunique()}")
print(f"Unique commodities: {prices['commodity'].nunique()}")
print(f"Unique admin1 regions: {prices['admin1'].nunique()}")

### 2.3 Exploratory Data Analysis

Exploratory analysis is used to understand the main patterns in the price and weather data.

We first examine the individual variables to understand their distributions, ranges, and data coverage. We then look at relationships between variables, with a focus on how weather conditions relate to commodity prices.

Maize and beans are the main commodities of interest because they are central to the business objectives of the project.


In [ ]:
# top commodities and markets by number of price observation -- univariate analysis
commodity_counts = prices['commodity'].value_counts().head(15)
market_counts = prices["market"].value_counts().head(15)

#plot
fig, axes = plt.subplots(1,2, figsize=(16,5))
commodity_counts.plot(
    kind="barh", 
    ax=axes[0], 
    color='green')

axes[0].set_title("Top 15 commodities by number of Price Observations")
axes[0].invert_yaxis() #invert y-axis of first plot only

market_counts.plot(
    kind="barh", 
    ax=axes[1], 
    color='blue')

axes[1].set_title("Top 15 Markets by number of price Observations")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


Maize and beans dominate the commodity coverage, while Nairobi and a handful of other major markets carry the largest number of price observations, confirming they are strong candidates for individual forecasting models.

In [ ]:
# price spread for the two priority commodities (Univariate)
priority_commodities = ["Maize", "Beans"]
subset = prices[prices["commodity"].isin(priority_commodities)]

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=subset, 
    x="commodity", 
    y="price", 
    palette=["green", "brown"],
    hue='commodity'
    )
plt.title("Price Distribution: Maize vs Beans (KES)")
plt.ylabel("Price (KES)")
plt.xlabel("Commodity")
plt.show()

Beans trade at a consistently higher price point than maize with a wider spread, which is expected given differing unit types across rows, this should be checked per unit before treating any values as outliers.

In [ ]:
# national average maize price by month (univariate)
maize = prices[prices["commodity"] == "Maize"].copy()
maize_monthly = maize.groupby(
    pd.Grouper(
        key="date", 
        freq="ME")
        )["price"].mean()

plt.figure(figsize=(14, 5))
maize_monthly.plot(color="green", linewidth=1.8)
plt.title("National Average Maize Price Over Time (Monthly Mean)")
plt.ylabel("Price (KES)")
plt.xlabel("Date")
plt.show()

Maize prices show a clear long term upward trend with repeated volatility spikes rather than a stable plateau, consistent with the structural price instability described in the Business Understanding section.

In [ ]:
# average maize price by calendar month, all years combined (univariate)
maize["month"] = maize["date"].dt.month
seasonal_avg = maize.groupby("month")["price"].mean()

plt.figure(figsize=(10, 5))
seasonal_avg.plot(kind="bar", color="orange")
plt.title("Average Maize Price by Calendar Month (All Years Combined)")
plt.xlabel("Month")
plt.ylabel("Average Price (KES)")
plt.xticks(rotation=0)
plt.show()

Average prices show a mild recurring seasonal shape across the calendar year, offering early, if not yet conclusive, support for Hypothesis H1 on harvest driven seasonality.

In [ ]:
# missing maize price coverage by market and year (univariate)
pivot_check = maize.pivot_table(
    index="market",
    columns=maize["date"].dt.year,
    values="price",
    aggfunc="mean"
)

plt.figure(figsize=(16, 8))
sns.heatmap(pivot_check.isna(), cbar=False, cmap="Reds")
plt.title("Missing Maize Price Data by Market and Year (Red = Missing)")
plt.xlabel("Year")
plt.ylabel("Market")
plt.show()

Coverage is uneven across markets, with some markets reporting nearly every year and others showing large red gaps, these sparse markets are candidates for exclusion or imputation in Data Preparation.

In [ ]:
# flag maize price outliers using the iqr method (univariate analysis)
q1 = maize["price"].quantile(0.25)
q3 = maize["price"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = maize[(maize["price"] < lower_bound) | (maize["price"] > upper_bound)]

In [ ]:
print(f"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}] KES")
print(f"Outlier observations detected: {len(outliers)} out of {len(maize)}")

In [ ]:
outliers[["date", "market", "price"]].sort_values("price", ascending=False).head(10)

### 2.4 Weather Data Exploration

Rainfall and temperature are the two variables most directly tied to agricultural output, and by extension to price formation several months later. Kenya has a  known bimodal rainfall pattern, long rains from March to May and short rains from October to December,which provides a natural benchmark against which the retrieved data can be validated.

In [ ]:
# daily rainfall and temperature for Nairobi, full year 2023 (univariate)
power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params_full_year = {
    "parameters": "T2M,PRECTOTCORR",
    "community": "ag",
    "longitude": nairobi_lon,
    "latitude": nairobi_lat,
    "start": "20230101",
    "end": "20231231",
    "format": "JSON",
}

response_year = requests.get(power_url, params=params_full_year, timeout=30)
weather_json = response_year.json()["properties"]["parameter"]

weather_df = pd.DataFrame({
    "date": pd.to_datetime(list(weather_json["T2M"].keys()), format="%Y%m%d"),
    "temperature": list(weather_json["T2M"].values()),
    "rainfall": list(weather_json["PRECTOTCORR"].values()),
})

In [ ]:
weather_df.set_index("date")[["rainfall"]].plot(figsize=(14, 4), color="blue")
plt.title("Daily Rainfall, Nairobi, 2023 (Checking for Bimodal Pattern)")
plt.ylabel("Rainfall (mm/day)")
plt.show()

Rainfall shows two distinct peak periods across the year, consistent with Kenya's known long rains and short rains pattern, which supports using this data as a credible exogenous feature.

### 2.5 Price and Weather Relationship

This section examines whether rainfall precedes price movement with a measurable lag, which is the central mechanic behind the early warning system. Since harvests occur several months after planting rains, the expected relationship is not simultaneous but lagged, meaning a drop in rainfall today should be tested against price levels several months in the future rather than against the current price.

In [ ]:
# test rainfall to price correlation at 0 to 6 month lags
monthly_rainfall = weather_df.set_index("date")["rainfall"].resample("ME").sum()
monthly_price = maize_monthly

combined = pd.DataFrame({"rainfall": monthly_rainfall, "price": monthly_price}).dropna()

lag_results = {}
for lag in range(0, 7):
    shifted_rainfall = combined["rainfall"].shift(lag)
    lag_results[lag] = shifted_rainfall.corr(combined["price"])

lag_series = pd.Series(lag_results)

In [ ]:
print("Correlation between rainfall (lagged) and maize price:")
print(lag_series.round(3))

In [ ]:
# rainfall to price lag correlation (Bivariate)
lag_series.plot(kind="bar", figsize=(8, 4), color="#6A1B9A")
plt.title("Rainfall to Price Lag Correlation (Months)")
plt.xlabel("Lag (months)")
plt.ylabel("Correlation coefficient")
plt.show()

This lag test currently uses only one calendar year of Nairobi weather against a multi year national price series, so the correlation values are indicative rather than conclusive, the proper version of this test happens once weather is pulled per market across the full date range in Data Preparation.

### 2.6 Coordinate Completeness

Initial planning assumed a separate markets reference file would be needed to map market names to coordinates for the weather join. Inspection of the price file itself showed this is unnecessary, latitude and longitude are already included directly in the main price dataset, one pair per market.

In [ ]:
# confirm lat/lon exist directly in the price data
coordinate_columns = [col for col in prices.columns if "lat" in col.lower() or "lon" in col.lower()]
coord_check = prices.groupby("market")[coordinate_columns].nunique()

In [ ]:
print("Coordinate related columns found:", coordinate_columns)
coord_check.head(10)

In [ ]:
# identify which markets are missing coordinates
missing_coords = prices[prices["latitude"].isna()]["market"].unique()

In [ ]:
print(f"Markets with missing coordinates: {len(missing_coords)}")
print(missing_coords)

### 2.7 Data Quality Summary

The data is generally suitable for analysis, but a few issues need to be addressed before modelling.

* **Commodity names:** Maize appears under different labels. These will need to be reviewed and consolidated where appropriate.
* **Duplicates:** No duplicate observations were found at the market, commodity, date, and price type level.
* **Outliers:** A small number of maize price observations were flagged using the IQR method. These will be reviewed rather than removed automatically.
* **Market coordinates:** One market is missing latitude and longitude and will be excluded from the weather join.
* **Date coverage:** The price data spans from January 2006 to August 2026, providing a long historical period for analysis.

These findings will guide the data cleaning and preparation steps before modelling.


### 2.8 Initial Insights and Transition to Data Preparation

The data is generally clean and suitable for the next stage. We found no duplicate observations, and almost all markets have the coordinates needed to connect them with weather data.

The rainfall analysis also showed a relationship between rainfall and prices at a four-month lag, providing early support for **H2**. This is an initial finding and will be tested further using the full dataset.

The main issue to resolve is the way maize is labelled. Different maize variants appear as separate commodities and may have different price scales and units. This needs to be handled before outlier detection and modelling.

These findings guide the cleaning, transformation, and feature engineering steps in the **Data Preparation** phase.


## 3. Data Preparation

This phase prepares the data for analysis and modelling based on the findings from Data Understanding.

The data will be cleaned, transformed, and combined with weather data. Commodity labels will be preserved as reported in the source data rather than merging products that may have different price behaviour.

We will also check market and commodity coverage before making decisions about which data to use for modelling.


### 3.1 Commodity Landscape

Before any cleaning, every commodity label in the dataset is reviewed, along with a simple flag distinguishing raw or staple products from processed derivatives such as flour or meal. This flag is for visibility only, it is not used to merge products together.

In [ ]:
commodity_overview = prices["commodity"].value_counts()
is_processed = prices["commodity"].str.contains("flour|meal|powder", case=False, na=False)
prices["is_processed"] = is_processed

In [ ]:
print(f"Total distinct commodity labels: {prices['commodity'].nunique()}")
print(f"Processed or derivative product rows: {is_processed.sum()}")
print(commodity_overview.head(30))

### 3.2 Unit Audit

Every distinct unit of measurement present across the full dataset is listed here, not assumed from maize alone. Weight based units are converted to a kilogram equivalent, non weight units, such as items sold per liter or per piece, are flagged separately rather than force converted.

In [ ]:
unit_counts = prices["unit"].value_counts()

In [ ]:
print(unit_counts)

In [ ]:
unit_to_kg = {
    "KG": 1,
    "90 KG": 90,
    "64 KG": 64,
    "50 KG": 50,
    "26 KG": 26,
    "126 KG": 126,
    "13 KG": 13,
    "200 G": 0.2,
    "400 G": 0.4,
}

prices["kg_equivalent"] = prices["unit"].map(unit_to_kg)
prices["price_per_kg"] = prices["price"] / prices["kg_equivalent"]

In [ ]:
unmapped_units = prices[prices["kg_equivalent"].isna()]["unit"].unique()
print(f"Unmapped units, genuinely non weight based: {unmapped_units}")
print(f"Rows converted to price per kg: {prices['price_per_kg'].notna().sum()} out of {len(prices)}")

### 3.3 Retail Price Selection

Retail prices are selected as the primary modeling series. This is a deliberate scoping decision, not an oversight, retail is the price point smallholder farmers, cooperatives, and urban consumers, the majority of beneficiary groups in the stakeholder analysis, actually transact on and can act upon directly. Wholesale price forecasting, most relevant to traders, NCPB, and millers, is a natural extension of this same pipeline, since wholesale rows remain intact in the source data and require no new data collection, only rerunning the completeness and modeling steps against `pricetype == "Wholesale"` instead. This is documented here as a defined next phase rather than an unaddressed gap.

In [ ]:
retail = prices[prices["pricetype"] == "Retail"].copy()
print(f"Retail rows: {len(retail)} out of {len(prices)} total rows")

In [ ]:
wholesale_rows = prices[prices["pricetype"] == "Wholesale"]
print(f"Wholesale rows available: {len(wholesale_rows)}")
print(f"Retail rows available: {len(retail)}")

### 3.4 Market and Commodity Completeness 

Completeness is computed for every market and commodity combination in the retail dataset at once, using each commodity's original label, with no merging assumptions. This gives a complete, honest picture of geographic and commodity coverage before any threshold is chosen.

In [ ]:
coverage = (
    retail
    .groupby(["market", "commodity"])["date"]
    .nunique()
    .reset_index(name="months_reported")
)

total_months_available = retail["date"].nunique()
coverage["completeness_pct"] = (coverage["months_reported"] / total_months_available * 100).round(1)

In [ ]:
print(f"Total market-commodity pairs: {len(coverage)}")
for threshold in [30, 40, 50, 60, 70]:
    qualifying = coverage[coverage["completeness_pct"] >= threshold]
    print(f"At {threshold} percent threshold: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

In [ ]:
# check the reporting span of each commodity
reporting_span = (
    retail
    .groupby(["market", "commodity"])["date"]
    .agg(first_reported="min", last_reported="max", months_reported="nunique")
    .reset_index()
)

reporting_span["active_months"] = (
    (reporting_span["last_reported"].dt.year - reporting_span["first_reported"].dt.year) * 12
    + (reporting_span["last_reported"].dt.month - reporting_span["first_reported"].dt.month)
    + 1
)

reporting_span["completeness_pct_fair"] = (reporting_span["months_reported"] / reporting_span["active_months"] * 100).round(1)

In [ ]:
print(reporting_span[["first_reported", "last_reported"]].describe())
for threshold in [50, 60, 70, 80, 90]:
    qualifying = reporting_span[reporting_span["completeness_pct_fair"] >= threshold]
    print(f"At {threshold} percent fair completeness: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

### 3.5 Assessing Historical Coverage

Having enough observations does not always mean having enough history for seasonal modelling. To identify a reliable yearly pattern, we need to see that pattern repeat across multiple years.

We therefore assess each market and commodity based on both **data completeness and the number of years of available history**. Market-commodity pairs with deeper histories are suitable for models such as Prophet, while pairs with shorter histories can be evaluated using LSTM or simpler baseline models.

This ensures that each model is used only where the available history is sufficient for it to perform reliably.


In [ ]:
reporting_span["years_active"] = ((reporting_span["last_reported"] - reporting_span["first_reported"]).dt.days / 365.25).round(1)

long_history = reporting_span[(reporting_span["completeness_pct_fair"] >= 60) & (reporting_span["years_active"] >= 3)]
recent_only = reporting_span[(reporting_span["completeness_pct_fair"] >= 60) & (reporting_span["years_active"] < 3)]

In [ ]:
print(f"Long history pairs (3+ years, 60pct fair completeness): {len(long_history)}, {long_history['market'].nunique()} markets, {long_history['commodity'].nunique()} commodities")
print(f"Recent only pairs (under 3 years, 60pct fair completeness): {len(recent_only)}, {recent_only['market'].nunique()} markets, {recent_only['commodity'].nunique()} commodities")

In [ ]:
for years in [1, 2, 3, 4]:
    qualifying = reporting_span[(reporting_span["completeness_pct_fair"] >= 60) & (reporting_span["years_active"] >= years)]
    print(f"At {years}+ years active: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

### 3.6 Final Market Commodity list

The long history and recent only groups together define the full modeling scope. Each retains its original commodity label, market, and price type, with no products merged.

In [ ]:
long_history["model_track"] = "prophet"
recent_only["model_track"] = "lstm"

shortlist = pd.concat([long_history, recent_only], ignore_index=True)[["market", "commodity", "model_track"]]

In [ ]:
print(f"Total shortlisted market-commodity pairs: {len(shortlist)}")
print(f"Unique markets: {shortlist['market'].nunique()}")
print(f"Unique commodities: {shortlist['commodity'].nunique()}")

In [ ]:
retail["kg_equivalent"] = retail["unit"].map(unit_to_kg)
retail["price_per_kg"] = retail["price"] / retail["kg_equivalent"]

modeling_data = retail.merge(shortlist, on=["market", "commodity"], how="inner")
modeling_data = modeling_data[modeling_data["price_per_kg"].notna()].copy()

In [ ]:
print(f"Modeling ready rows: {len(modeling_data)}")
print(f"Rows dropped for non weight units: {len(retail.merge(shortlist, on=['market','commodity'])) - len(modeling_data)}")

### 3.7 Outlier Recheck

Outlier bounds are recalculated on the corrected, unit standardized price per kilogram field, computed separately per commodity, since pooling different commodities together, the same mistake that distorted the earlier maize check, would produce meaningless bounds here too.

In [ ]:
def flag_outliers(group):
    q1, q3 = group["price_per_kg"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return group[(group["price_per_kg"] < lower) | (group["price_per_kg"] > upper)]

outliers_by_commodity = (
    modeling_data
    .groupby("commodity", group_keys=True)
    .apply(flag_outliers, include_groups=False)
    .reset_index(level=0)
)

In [ ]:
print(f"Total outliers flagged: {len(outliers_by_commodity)} out of {len(modeling_data)}")
print(outliers_by_commodity["commodity"].value_counts())

### 3.8 Outlier Verification in salt commodity.

Manual inspection of the salt outliers shows a tight, plausible price cluster around 90 to 115 KES per kilogram, with a small tail extending toward 275 KES per kilogram concentrated in refugee camp markets such as Kakuma, Daadab, and Kalobeyei, where transport and supply chain costs are known to be higher. These are genuine price observations rather than conversion errors, and all flagged outliers across every commodity are retained in the dataset rather than removed, since the project's own anomaly detection layer is designed to act on exactly this kind of divergence.

In [ ]:
salt_outliers = outliers_by_commodity[outliers_by_commodity["commodity"] == "Salt"]
print(salt_outliers[["market", "date", "price", "unit", "price_per_kg"]].to_string())

### 3.9 Outlier Persistence Classification

An IQR flag alone could not distinguish a genuine market shift, where prices settle at a new level and stay there, from a transient spike that reverts, or a data entry error that appears once and never repeats. Each flagged outlier is classified by comparing the price level in the months immediately after the flagged date against the baseline level in the months immediately before it. If the new level persists, it is treated as a genuine shift worth investigating further. If it reverts close to baseline, it is treated as a transient spike, a real but temporary event. This same logic is the direct precursor to the residual based anomaly detection layer planned for the final system, since both rely on comparing actual behavior against a recent baseline.

In [ ]:
def classify_outlier(row, data, window=2):
    series = data[(data["market"] == row["market"]) & (data["commodity"] == row["commodity"])].sort_values("date")
    match = series[series["date"] == row["date"]]
    if match.empty:
        return "unknown"
    pos = series.index.get_loc(match.index[0])
    before = series.iloc[max(0, pos - window):pos]["price_per_kg"]
    after = series.iloc[pos + 1: pos + 1 + window]["price_per_kg"]
    if before.empty or after.empty:
        return "insufficient surrounding data"
    baseline = before.mean()
    reverted = abs(after.mean() - baseline) < abs(row["price_per_kg"] - baseline) * 0.5
    return "transient spike" if reverted else "persistent shift"

outliers_by_commodity["classification"] = outliers_by_commodity.apply(
    lambda row: classify_outlier(row, modeling_data), axis=1
)

In [ ]:
print(outliers_by_commodity["classification"].value_counts())

In [ ]:
persistent_shifts = outliers_by_commodity[outliers_by_commodity["classification"] == "persistent shift"]
persistent_shifts[["market", "commodity", "date", "price_per_kg"]].sort_values("date")

### 3.10 Handling Outliers

No flagged outliers are removed from the modeling dataset. Transient spikes are retained as genuine historical events and reserved as informal validation cases for the anomaly detection layer built later in the project. Persistent shifts are retained in the modeling data but a sample was manually reviewed to rule out unit or entry errors masquerading as real market changes, since a genuine shift and a silent data bug can produce an identical pattern in this test. Rows with insufficient surrounding data remain in the dataset but are not treated as evidence of either category, since there is not enough history on one side to judge them fairly.

### 3.10 Weather Retrieval at Scale

With the market and commodity shortlist finalized, daily rainfall and temperature data is retrieved from the NASA POWER API for every shortlisted market's coordinates, covering the full date range needed for modeling. A short pause between requests avoids triggering rate limiting on the API, given the number of markets involved.

In [ ]:
import time

market_coords = modeling_data[["market", "latitude", "longitude"]].drop_duplicates()
weather_records = []

for _, row in market_coords.iterrows():
    params_loop = {
        "parameters": "T2M,PRECTOTCORR",
        "community": "ag",
        "longitude": row["longitude"],
        "latitude": row["latitude"],
        "start": "20060101",
        "end": "20260815",
        "format": "JSON",
    }
    resp = requests.get(power_url, params=params_loop, timeout=60)
    if resp.status_code == 200:
        param_data = resp.json()["properties"]["parameter"]
        df_market = pd.DataFrame({
            "date": pd.to_datetime(list(param_data["T2M"].keys()), format="%Y%m%d"),
            "temperature": list(param_data["T2M"].values()),
            "rainfall": list(param_data["PRECTOTCORR"].values()),
        })
        df_market["market"] = row["market"]
        weather_records.append(df_market)
    time.sleep(1)

weather_all = pd.concat(weather_records, ignore_index=True)

In [ ]:
print(f"Markets retrieved: {weather_all['market'].nunique()} out of {len(market_coords)}")
print(f"Total daily weather rows: {len(weather_all)}")

### 3.11 Weather Aggregation to Monthly

Price data is recorded monthly, while the retrieved weather data is daily, so weather is resampled to a monthly grain per market before joining, rainfall summed and temperature averaged, matching how each variable naturally aggregates over a month.

In [ ]:
weather_monthly = (
    weather_all
    .set_index("date")
    .groupby("market")
    .resample("ME")
    .agg({"rainfall": "sum", "temperature": "mean"})
    .reset_index()
)

In [ ]:
print(f"Monthly weather rows: {len(weather_monthly)}")
print(f"Markets represented: {weather_monthly['market'].nunique()}")

### 3.12 Lagged Weather Feature Engineering

The lag correlation test in Data Understanding found the strongest relationship between rainfall and maize price at a four month lag, with a secondary signal at three months. Both lags are engineered as explicit features here, computed per market so no market's lag mixes with another's history.

In [ ]:
weather_monthly = weather_monthly.sort_values(["market", "date"])
weather_monthly["rainfall_lag_3"] = weather_monthly.groupby("market")["rainfall"].shift(3)
weather_monthly["rainfall_lag_4"] = weather_monthly.groupby("market")["rainfall"].shift(4)
weather_monthly["temperature_lag_3"] = weather_monthly.groupby("market")["temperature"].shift(3)
weather_monthly["temperature_lag_4"] = weather_monthly.groupby("market")["temperature"].shift(4)

### 3.13 Price and Weather Integration
Cleaned monthly prices are joined to the lagged monthly weather features on market and date. The weather table's own date column is dropped before the merge, since both tables otherwise carry a column named date, which would silently rename both to date_x and date_y and break every step downstream that references date directly.

In [ ]:
modeling_data["date_month"] = modeling_data["date"].values.astype("datetime64[M]")
weather_monthly["date_month"] = weather_monthly["date"].values.astype("datetime64[M]")
weather_features = weather_monthly.drop(columns=["date"])

master = modeling_data.merge(
    weather_features,
    on=["market", "date_month"],
    how="left"
)

In [ ]:
print(f"Master table rows: {len(master)}")
print(f"Rows with matched weather data: {master['rainfall'].notna().sum()}")

### 3.14 Full Scale Lag Validation

The preliminary lag correlation in Data Understanding compared one year of Nairobi rainfall against a twenty year national price series, a temporal mismatch flagged at the time as a limitation. This section repeats the test properly, using the master table, where price and weather share the same market and the same date for every row, so the comparison is genuinely apples to apples across the full history and every shortlisted market.

In [ ]:
rainfall_corr_3 = master["price_per_kg"].corr(master["rainfall_lag_3"])
rainfall_corr_4 = master["price_per_kg"].corr(master["rainfall_lag_4"])
temp_corr_3 = master["price_per_kg"].corr(master["temperature_lag_3"])
temp_corr_4 = master["price_per_kg"].corr(master["temperature_lag_4"])

In [ ]:
print(f"Rainfall lag 3 months, correlation with price: {rainfall_corr_3:.3f}")
print(f"Rainfall lag 4 months, correlation with price: {rainfall_corr_4:.3f}")
print(f"Temperature lag 3 months, correlation with price: {temp_corr_3:.3f}")
print(f"Temperature lag 4 months, correlation with price: {temp_corr_4:.3f}")

### 3.15 Train, Validation, Test Split

Because this is time series data, the split is chronological rather than random, reserving the most recent months as a genuine holdout so forecast accuracy reflects real forward looking performance rather than leakage from future information.

In [ ]:
master = master.sort_values("date")
cutoff_val = master["date"].quantile(0.8)
cutoff_test = master["date"].quantile(0.9)

train = master[master["date"] < cutoff_val]
val = master[(master["date"] >= cutoff_val) & (master["date"] < cutoff_test)]
test = master[master["date"] >= cutoff_test]

In [ ]:
print(f"Train: {len(train)} rows, up to {train['date'].max()}")
print(f"Validation: {len(val)} rows, {val['date'].min()} to {val['date'].max()}")
print(f"Test: {len(test)} rows, from {test['date'].min()}")

### 3.16 Data Preparation Summary

Data Preparation addressed three structural issues surfaced during cleaning. Commodity labels were kept distinct rather than merged, since WFP records different products, such as raw grain and flour, under separate labels that carry genuinely different price behavior. Completeness was measured against each market commodity pair's own active reporting window rather than the full dataset span, since most series only began consistent reporting in late 2023, and a fixed span measurement unfairly penalized commodities and markets that started later. Units were standardized to a common price per kilogram basis, with weight based units, including gram denominations initially missed, converted correctly, and genuinely non weight units, such as liters and countable items, left unconverted and excluded from weight based analysis.

The resulting shortlist covers 871 market commodity pairs across 161 markets and 36 commodities, split by history depth into 130 pairs with three or more years of history suited to Prophet's seasonal modeling, and 741 pairs with shorter but consistent history suited to the LSTM. Statistical outliers were identified per commodity rather than pooled, and retained in the dataset rather than removed, since the project's anomaly detection layer is designed to act on genuine price divergence rather than treat it as noise to discard. Weather data was retrieved for 160 of 161 markets from the NASA POWER API, aggregated to a monthly grain, and joined to price data using a three and four month rainfall lag, based on the correlation pattern found in Data Understanding. The resulting master table contains 7,534 rows with a 99.1 percent weather match rate, split chronologically into train, validation, and test sets to support honest time series backtesting in the Modeling phase.

## 4. Modelling

### 4.1 Modelling Objective

The goal of this phase is to build and compare models that can forecast food prices for individual market and commodity combinations.

Three forecasting approaches will be evaluated:

* **Naive Forecasting**, used as a simple baseline. It assumes that the next price will be similar to the most recently observed price and provides a minimum benchmark for model performance.
* **Prophet**, used for market and commodity pairs with sufficient historical data and clear time-based patterns.
* **LSTM**, used as a neural network approach to capture more complex patterns in the time series.

The target variable is `price_per_kg`, representing the standardized commodity price in Kenyan Shillings per kilogram. Lagged weather variables, particularly rainfall and temperature, will be included as additional features where appropriate.

Since the data is time dependent, the chronological train, validation, and test splits created during Data Preparation will be preserved. All models will be evaluated on unseen data using **MAE and MAPE**.

The final model will be selected based on forecasting performance, stability, interpretability, and suitability for deployment in the Kenya Food Price Early Warning System.


In [ ]:
#import libraries used in modeling

# Prophet forecasting model
from prophet import Prophet

# LSTM neural network components
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Training callbacks
from tensorflow.keras.callbacks import EarlyStopping

# Feature scaling
from sklearn.preprocessing import MinMaxScaler

# Forecast evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

# warnings
import warnings
warnings.filterwarnings('ignore')

# Define the primary forecasting horizon.
# Because the dataset is monthly, one period represents approximately one month.

FORECAST_HORIZON = 1



### 4.2 Forecasting Horizon

The forecasting horizon is determined by the resolution of the data available to the system. After integrating the WFP price data with the NASA POWER weather data, the modelling dataset is structured at a **monthly frequency**. This means that forecasting prices at a weekly level would require assumptions or transformations that are not directly supported by the original data.

The project therefore focuses on predicting commodity prices **one to three months ahead**. The **one-month horizon will be the primary forecasting target**, as it provides a practical early-warning window while remaining close enough to the observed data to support reliable predictions.

The **two and three-month horizons** will be evaluated as additional forecasting experiments. These longer horizons will help determine how quickly forecast accuracy decreases as the prediction moves further into the future.

This represents a data-driven refinement of the original two-to-four-week objective. The business goal remains the same: **provide early visibility of potential food price changes before they become larger problems.** The forecasting horizon has simply been aligned with the actual temporal resolution of the data.


### 4.3 Naive Baseline Model

A naive forecasting model is established before training more sophisticated models.

The baseline assumes that the next month's price will be equal to the most recently observed price. Although this approach is simple, it provides an important benchmark for determining whether Prophet and LSTM actually provide additional predictive value.

A sophisticated model should only be considered useful if it improves upon this simple benchmark.


In [ ]:
# Sort the data chronologically within each market-commodity series

baseline_data = master.sort_values(
    ["market", "commodity", "date"]
).copy()

# Create the previous month's price for each market-commodity pair.
# This prevents information from one market or commodity leaking into another.

baseline_data["naive_prediction"] = (
    baseline_data
    .groupby(["market", "commodity"])["price_per_kg"]
    .shift(1)
)

# Keep only rows where both the actual price and previous price exist.
baseline_test = baseline_data.dropna(
    subset=["price_per_kg", "naive_prediction"]
).copy()

# Calculate baseline errors.
naive_mae = mean_absolute_error(
    baseline_test["price_per_kg"],
    baseline_test["naive_prediction"]
)

# MAPE can become unstable when actual prices are zero.
# The dataset should therefore be protected against division by zero.
non_zero = baseline_test["price_per_kg"] != 0

naive_mape = np.mean(
    np.abs(
        (
            baseline_test.loc[non_zero, "price_per_kg"]
            - baseline_test.loc[non_zero, "naive_prediction"]
        )
        / baseline_test.loc[non_zero, "price_per_kg"]
    )
) * 100



In [ ]:
print(f"Naive Baseline MAE: {naive_mae:.2f}")
print(f"Naive Baseline MAPE: {naive_mape:.2f}%")

### 4.4 Initial Modelling Series

The integrated dataset contains many market and commodity combinations, but they do not all have the same amount of historical data. Before applying the forecasting pipeline across all series, we first select a well-populated series to develop and validate the modelling workflow.

Based on the available history in the master dataset, **Kitui maize (white)** provides a strong starting point. It contains **180 monthly observations**, giving the models enough historical data to learn patterns while also providing a consistent series for testing the forecasting pipeline.

Kitui maize is therefore used as the initial modelling series. Once the pipeline has been validated, the same process can be applied to the remaining market and commodity combinations with sufficient historical coverage.

This approach allows the modelling workflow to be tested on a representative, data-rich series before being scaled across the wider dataset.


In [ ]:
# Select the market and commodity with sufficient historical coverage
market = "Kitui"
commodity = "Maize (white)"

series = master[
    (master["market"] == market) &
    (master["commodity"] == commodity)
].copy()

series = series.sort_values("date")

series[
    [
        "date",
        "market",
        "commodity",
        "price_per_kg",
        "rainfall_lag_3",
        "rainfall_lag_4",
        "temperature_lag_3",
        "temperature_lag_4"
    ]
].tail()

### 4.5 Price Series for the Selected Market

Before training the forecasting model, the selected market–commodity series is visualized to confirm that the time series contains sufficient observations and to identify broad patterns such as trends, seasonal movements, sudden increases, and decreases.

This visualization also provides a visual reference for interpreting the forecasts produced by the models.


In [ ]:
# Plot the historical price series
plt.figure(figsize=(12, 5))
plt.plot(
    series["date"],
    series["price_per_kg"],
    label="Actual Price"
)
plt.title(
    f"{commodity} Price in {market}"
)
plt.xlabel("Date")
plt.ylabel("Price per kg (KES)")
plt.legend()
plt.show()

### 4.6 Data preparaion for prophet

Prophet requires a dataframe containing a date column named `ds` and a target variable named `y`.

The standardized `price_per_kg` variable is therefore renamed to `y`, while the observation date is renamed to `ds`.

The weather variables are retained as additional regressors so that their contribution to price forecasting can be evaluated.


In [ ]:
# Prepare the selected series for Prophet

prophet_data = series[
    [
        "date",
        "price_per_kg",
        "rainfall_lag_3",
        "rainfall_lag_4",
        "temperature_lag_3",
        "temperature_lag_4"
    ]
].copy()

# Rename columns according to Prophet's requirements
prophet_data = prophet_data.rename(
    columns={
        "date": "ds",
        "price_per_kg": "y"
    }
)

# Remove rows where weather regressors are unavailable.
# This is important because Prophet cannot train with missing regressor values.
prophet_data = prophet_data.dropna()

prophet_data.head()

### 4.7 Prophet Train, Validation and Test Sets

The chronological structure of the data is preserved during model development.

The training set is used to estimate model parameters, the validation set is used for model development and parameter decisions, and the test set remains untouched until final evaluation.

This prevents future observations from influencing the model during training and provides a more realistic estimate of how the forecasting system will perform after deployment.


In [ ]:
# Convert the series-specific chronological cut-off dates into timestamps
# Use quantiles based on the filtered 'series' data, not the global 'master' data

series_train_end = series["date"].quantile(0.8)
series_val_end = series["date"].quantile(0.9)

# Create chronological splits for the selected Prophet series

prophet_train = prophet_data[
    prophet_data["ds"] <= series_train_end
].copy()

prophet_val = prophet_data[
    (prophet_data["ds"] > series_train_end) &
    (prophet_data["ds"] <= series_val_end)
].copy()

prophet_test = prophet_data[
    prophet_data["ds"] > series_val_end
].copy()



In [ ]:
print("Training observations:", len(prophet_train))
print("Validation observations:", len(prophet_val))
print("Test observations:", len(prophet_test))

### 4.8 Prophet Baseline Model

Prophet is used as the primary baseline forecasting model because it is designed for time series containing trend and seasonal patterns and provides interpretable forecasts.

The model will first be trained using historical price information. Lagged weather variables are then added as external regressors because the Data Understanding phase investigated rainfall and temperature as potential drivers of food price movements.

The model is intentionally kept relatively simple at this stage. Hyperparameter tuning will only be considered after establishing a reliable baseline.


In [ ]:
# Create the Prophet model

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

# Add lagged weather variables as external regressors.
# These variables were created during Data Preparation.

prophet_model.add_regressor("rainfall_lag_3")
prophet_model.add_regressor("rainfall_lag_4")
prophet_model.add_regressor("temperature_lag_3")
prophet_model.add_regressor("temperature_lag_4")

# Train the model using only the training data
prophet_model.fit(prophet_train)

### 4.9 Prophet Validation

The trained Prophet model is evaluated on the validation period to determine how accurately it predicts observations that were not included during training.

The validation results will be used to identify whether the model configuration is adequate before the final test evaluation.


In [ ]:
# Generate predictions for the validation period

prophet_val_forecast = prophet_model.predict(
    prophet_val[
        [
            "ds",
            "rainfall_lag_3",
            "rainfall_lag_4",
            "temperature_lag_3",
            "temperature_lag_4"
        ]
    ]
)

# Combine predictions with actual prices

prophet_val_results = prophet_val[
    ["ds", "y"]
].copy()

prophet_val_results["prediction"] = (
    prophet_val_forecast["yhat"].values
)

# Calculate MAE
val_mae = mean_absolute_error(
    prophet_val_results["y"],
    prophet_val_results["prediction"]
)

# Calculate MAPE safely
non_zero = prophet_val_results["y"] != 0

val_mape = np.mean(
    np.abs(
        (
            prophet_val_results.loc[non_zero, "y"]
            - prophet_val_results.loc[non_zero, "prediction"]
        )
        / prophet_val_results.loc[non_zero, "y"]
    )
) * 100

print(f"Prophet Validation MAE: {val_mae:.2f}")
print(f"Prophet Validation MAPE: {val_mape:.2f}%")

In [ ]:
# Plot Prophet Validation Forecast
# Plot actual versus predicted validation prices

plt.figure(figsize=(12, 5))

plt.plot(
    prophet_val_results["ds"],
    prophet_val_results["y"],
    label="Actual"
)

plt.plot(
    prophet_val_results["ds"],
    prophet_val_results["prediction"],
    label="Prophet Forecast"
)

plt.title(
    f"Prophet Validation Forecast: {commodity} - {market}"
)

plt.xlabel("Date")
plt.ylabel("Price per kg (KES)")
plt.legend()
plt.show()

### 4.10 Final Prophet Test Evaluation

After the Prophet configuration has been established using the training and validation periods, the model is evaluated on the test period.

The test set represents the most recent unseen observations and therefore provides the most realistic estimate of the model's expected forecasting performance.

The test results will later be compared against the naive baseline and LSTM model.


In [ ]:
# Generate forecasts for the untouched test period

prophet_test_forecast = prophet_model.predict(
    prophet_test[
        [
            "ds",
            "rainfall_lag_3",
            "rainfall_lag_4",
            "temperature_lag_3",
            "temperature_lag_4"
        ]
    ]
)

# Store actual and predicted values together

prophet_test_results = prophet_test[
    ["ds", "y"]
].copy()

prophet_test_results["prediction"] = (
    prophet_test_forecast["yhat"].values
)

# Calculate MAE
prophet_test_mae = mean_absolute_error(
    prophet_test_results["y"],
    prophet_test_results["prediction"]
)

# Calculate MAPE
non_zero = prophet_test_results["y"] != 0

prophet_test_mape = np.mean(
    np.abs(
        (
            prophet_test_results.loc[non_zero, "y"]
            - prophet_test_results.loc[non_zero, "prediction"]
        )
        / prophet_test_results.loc[non_zero, "y"]
    )
) * 100



In [ ]:
print(f"Prophet Test MAE: {prophet_test_mae:.2f}")
print(f"Prophet Test MAPE: {prophet_test_mape:.2f}%")

In [ ]:
# plot the final prophet forecast
# Plot actual prices against Prophet predictions

plt.figure(figsize=(12, 5))

plt.plot(
    prophet_test_results["ds"],
    prophet_test_results["y"],
    label="Actual"
)

plt.plot(
    prophet_test_results["ds"],
    prophet_test_results["prediction"],
    label="Prophet"
)

plt.title(
    f"Prophet Test Forecast: {commodity} - {market}"
)

plt.xlabel("Date")
plt.ylabel("Price per kg (KES)")
plt.legend()
plt.show()

## 4.13 LSTM Forecasting Model

### 4.13.1 LSTM Modelling Objective

Long Short-Term Memory (LSTM) networks are recurrent neural networks designed to learn patterns from sequential data.

The LSTM model will be used as a comparison model to determine whether a neural-network-based approach can capture price dynamics that are not adequately represented by the Prophet baseline.

The model will use historical price and lagged weather variables as input features. The observations will be transformed into rolling sequences so that the model learns from a fixed number of previous months when predicting the next month's price.

The LSTM will be evaluated using the same chronological validation and test periods used for Prophet to ensure a fair comparison.


### 4.14 Prepare LSTM Features

In [ ]:
# Select features that will be supplied to the LSTM

lstm_features = [
    "price_per_kg",
    "rainfall_lag_3",
    "rainfall_lag_4",
    "temperature_lag_3",
    "temperature_lag_4"
]

lstm_data = series[
    ["date"] + lstm_features
].copy()

# Remove rows containing missing modelling values
lstm_data = lstm_data.dropna()

# Sort chronologically
lstm_data = lstm_data.sort_values("date")

lstm_data.head()